# Ensemble MLP Tutorial

Three uncertainty-aware surrogate modes on GFP brightness prediction:

- **Seed ensemble** — N independently-seeded networks
- **MC dropout** — one network, T stochastic inference passes
- **Combined** — N networks × T passes

Each section trains one variant, predicts, and plots. Section 6 compares all three.

In [ ]:
import time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
import torch

from alf_core import BaseDatasetConfig, Candidate, LabelledCandidates, Modality
from alf_tools.datasets.gfp import GFP
from alf_tools.models.mlp import MLPModel, MLPModelConfig, MLPTrainConfig
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig

logging.basicConfig(level=logging.INFO, format="%(message)s")

## 1. Setup

In [ ]:
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
FAST_MODE   = DEVICE == "cpu"
N_MEMBERS   = 10
N_EPOCHS    = 25 if FAST_MODE else 50
N_MC_PASSES = 10 if FAST_MODE else 20
BASE_SEED   = 42
DROPOUT_P   = 0.1
HIDDEN_DIMS = [128, 64]

print(f"Device : {DEVICE}")
print(f"Mode   : {'FAST (CPU)' if FAST_MODE else 'FULL (GPU)'}")
print(f"N_MEMBERS={N_MEMBERS}  N_EPOCHS={N_EPOCHS}  N_MC_PASSES={N_MC_PASSES}")

## 2. GFP Dataset

GFP (green fluorescent protein) is a standard benchmark for sequence-fitness modelling. Each of the 1 000 variants is a 237-nt nucleotide sequence; the label is median brightness. Sequences cluster around wild-type, making calibration informative: a well-calibrated model should be more uncertain on distant variants.

In [ ]:
gfp = GFP(BaseDatasetConfig())
labelled = gfp.load_dataset()

rng = np.random.default_rng(BASE_SEED)
indices = rng.permutation(len(labelled))
n_train = int(0.8 * len(labelled))
train_idx, val_idx = indices[:n_train], indices[n_train:]
train_raw_cands, train_labels = labelled[train_idx]
val_raw_cands,   val_labels   = labelled[val_idx]

print(f"Sequences : {len(labelled)}")
print(f"Train : {len(train_labels)} | Val : {len(val_labels)}")
print(f"Brightness range : [{labelled.labels.min():.3f}, {labelled.labels.max():.3f}]  mean={labelled.labels.mean():.3f}")

In [ ]:
# MLPModel only accepts TABULAR/EMBEDDING modality; one-hot-encode the nucleotide sequences.
NUCLEOTIDES = list("ACGT")
SEQ_LEN     = len(labelled.candidates[0].data)
INPUT_DIM   = SEQ_LEN * 4

def one_hot_encode(seq: str) -> np.ndarray:
    arr = np.zeros(len(seq) * 4, dtype=np.float32)
    for i, nuc in enumerate(seq):
        arr[i * 4 + NUCLEOTIDES.index(nuc)] = 1.0
    return arr

def to_tabular(raw_cands: list) -> list:
    return [Candidate(data=one_hot_encode(c.data), modality=Modality.TABULAR) for c in raw_cands]

train_cands = to_tabular(train_raw_cands)
val_cands   = to_tabular(val_raw_cands)
train_data  = LabelledCandidates(candidates=train_cands, labels=train_labels)
val_data    = LabelledCandidates(candidates=val_cands,   labels=val_labels)

print(f"Seq len: {SEQ_LEN} | Input dim: {INPUT_DIM}")